# **CELL 1 — Mount Drive & Setup Paths**

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/Jigsaw_Bias_Project/Datasets/train.csv'

if os.path.exists(DATA_PATH):
    print(" Dataset found.")
else:
    print(" train.csv not found! Check the path.")

Mounted at /content/drive
✅ Dataset found.


# **CELL 2 — Imports & Deep Cleaning Function**

In [ ]:
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def deep_clean_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\w*\d\w*', '', text)
    words = text.split()
    cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(cleaned_words)

# **CELL 3 — Load, Clean, and Split Data**

In [ ]:
print("Loading Data...")
train = pd.read_csv(DATA_PATH, nrows=100000)
train['is_toxic'] = (train['target'] >= 0.5).astype(int)

print("Cleaning Text... (Takes ~1 minute)")
train['clean_text'] = train['comment_text'].apply(deep_clean_text)

print("Balancing Dataset (50/50)...")
toxic_df = train[train['is_toxic'] == 1]
clean_df = train[train['is_toxic'] == 0].sample(n=len(toxic_df), random_state=42)
balanced_df = pd.concat([toxic_df, clean_df]).sample(frac=1, random_state=42).reset_index(drop=True)

# Split into X and y
X = balanced_df['clean_text']
y = balanced_df['is_toxic']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f" Data Ready for BERT! Train size: {len(X_train)}, Test size: {len(X_test)}")

Loading Data...
Cleaning Text... (Takes ~1 minute)
Balancing Dataset (50/50)...
✅ Data Ready for BERT! Train size: 11064, Test size: 2766


# **CELL 4 — The Fairness Strategy: Sample Weighting**

In [ ]:
import numpy as np

print("Calculating Bias Mitigation Weights...")

identity_cols = [
    'male', 'female', 'homosexual_gay_or_lesbian',
    'christian', 'jewish', 'muslim', 'black', 'white'
]

balanced_df['sample_weight'] = 1.0

identity_mask = balanced_df[identity_cols].ge(0.5).any(axis=1)
balanced_df.loc[identity_mask, 'sample_weight'] = 3.0

# 4. Extract the weights for our train and test splits using their original indices
train_weights = balanced_df.loc[X_train.index, 'sample_weight'].tolist()
test_weights = balanced_df.loc[X_test.index, 'sample_weight'].tolist()

print(f"Weights applied! {identity_mask.sum()} identity comments are now prioritized.")

Calculating Bias Mitigation Weights...
Weights applied! 1430 identity comments are now prioritized.


# **CELL 5 — Hugging Face Tokenization & Dataset Formatting**

In [ ]:
!pip install -q transformers datasets

from transformers import DistilBertTokenizerFast
from datasets import Dataset
import pandas as pd

print("Loading DistilBERT Tokenizer...")
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

print("Injecting weights into Hugging Face Datasets...")
train_df = pd.DataFrame({'text': X_train, 'labels': y_train, 'weights': train_weights}).reset_index(drop=True)
test_df = pd.DataFrame({'text': X_test, 'labels': y_test, 'weights': test_weights}).reset_index(drop=True)

hf_train = Dataset.from_pandas(train_df)
hf_test = Dataset.from_pandas(test_df)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

print("Tokenizing... (This takes about 1-2 minutes)")
tokenized_train = hf_train.map(tokenize_function, batched=True)
tokenized_test = hf_test.map(tokenize_function, batched=True)

print("Data formatted for the Transformer!")

Loading DistilBERT Tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Injecting weights into Hugging Face Datasets...
Tokenizing... (This takes about 1-2 minutes)


Map:   0%|          | 0/11064 [00:00<?, ? examples/s]

Map:   0%|          | 0/2766 [00:00<?, ? examples/s]

Data formatted for the Transformer!


# **CELL 6 — The Bias-Aware Custom Trainer & Training Execution**

In [ ]:
from transformers import DistilBertForSequenceClassification, TrainingArguments, Trainer
import torch
from torch import nn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


# 1. THE CUSTOM LOSS FUNCTION
class BiasMitigationTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        # Extract our custom weights
        weights = inputs.pop("weights", None)

        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Calculate standard error
        loss_fct = nn.CrossEntropyLoss(reduction='none')
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        # Multiply the error by our 3x penalty if it's an identity comment
        if weights is not None:
            loss = (loss * weights).mean()
        else:
            loss = loss.mean()

        return (loss, outputs) if return_outputs else loss

# 2. EVALUATION METRICS

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}


# 3. INITIALIZE AND TRAIN

print("Loading DistilBERT Architecture...")
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"Hardware accelerator detected: {device.type.upper()}")

training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/Jigsaw_Bias_Project/Models/bert_mitigated',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,                       # Speeds up training on T4 GPUs
    remove_unused_columns=False,
    report_to="none"
)

trainer = BiasMitigationTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics
)

print("\nStarting Deep Learning Training with Bias Penalties...")
trainer.train()

Loading DistilBERT Architecture...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Hardware accelerator detected: CUDA

Starting Deep Learning Training with Bias Penalties...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.594397,0.416935,0.870933,0.872454,0.840909,0.906459
2,0.380467,0.458622,0.868040,0.870796,0.832206,0.913140


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1384, training_loss=0.43992488508279615, metrics={'train_runtime': 244.0605, 'train_samples_per_second': 90.666, 'train_steps_per_second': 5.671, 'total_flos': 732809649364992.0, 'train_loss': 0.43992488508279615, 'epoch': 2.0})

# **CELL 7 — Generate BERT Predictions**

In [ ]:
import numpy as np
from scipy.special import softmax

print("Generating predictions on the test set...")
predictions = trainer.predict(tokenized_test)

# Convert logits to probabilities
probs = softmax(predictions.predictions, axis=1)

# Extract probabilities for the 'Toxic' class (index 1) and create binary labels
y_pred_bert_prob = probs[:, 1]
y_pred_bert = (y_pred_bert_prob >= 0.5).astype(int)

print("Predictions generated!")

Generating predictions on the test set...


Predictions generated!


# **CELL 8 — The Final BERT Fairness Audit**

In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd

def compute_fpr_fnr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    return fpr, fnr

def classify_gap(gap):
    if gap < 0.02: return "Minimal Bias"
    elif 0.02 <= gap <= 0.05: return "Moderate Concern"
    else: return "Serious Bias Signal"

print("--- BERT PHASE 3 FAIRNESS AUDIT ---")

test_data = train.loc[X_test.index]
y_pred_series = pd.Series(y_pred_bert, index=y_test.index)

for identity in identity_cols:
    subgroup = test_data[test_data[identity] == 1]
    background = test_data[test_data[identity] == 0]

    y_true_sub = y_test.loc[subgroup.index]
    y_pred_sub = y_pred_series.loc[subgroup.index]

    y_true_back = y_test.loc[background.index]
    y_pred_back = y_pred_series.loc[background.index]

    fpr_sub, fnr_sub = compute_fpr_fnr(y_true_sub, y_pred_sub)
    fpr_back, fnr_back = compute_fpr_fnr(y_true_back, y_pred_back)

    fpr_gap = abs(fpr_sub - fpr_back)
    fnr_gap = abs(fnr_sub - fnr_back)

    print(f"\n--- {identity.upper()} ---")
    print(f"FPR Gap: {fpr_gap:.4f} → {classify_gap(fpr_gap)}")
    print(f"FNR Gap: {fnr_gap:.4f} → {classify_gap(fnr_gap)}")

--- BERT PHASE 3 FAIRNESS AUDIT ---

--- MALE ---
FPR Gap: 0.0428 → Moderate Concern
FNR Gap: 0.0017 → Minimal Bias

--- FEMALE ---
FPR Gap: 0.0959 → Serious Bias Signal
FNR Gap: 0.0425 → Moderate Concern

--- HOMOSEXUAL_GAY_OR_LESBIAN ---
FPR Gap: 0.1802 → Serious Bias Signal
FNR Gap: 0.0642 → Serious Bias Signal

--- CHRISTIAN ---
FPR Gap: 0.1465 → Serious Bias Signal
FNR Gap: 0.0667 → Serious Bias Signal

--- JEWISH ---
FPR Gap: 0.1481 → Serious Bias Signal
FNR Gap: 0.0686 → Serious Bias Signal

--- MUSLIM ---
FPR Gap: 0.0651 → Serious Bias Signal
FNR Gap: 0.0688 → Serious Bias Signal

--- BLACK ---
FPR Gap: 0.0480 → Moderate Concern
FNR Gap: 0.0728 → Serious Bias Signal

--- WHITE ---
FPR Gap: 0.0200 → Minimal Bias
FNR Gap: 0.0489 → Moderate Concern


# **CELL 9 — Save the BERT Model & Tokenizer**

In [ ]:
import os

print("Saving the BERT Brain to Shared Drive...")
output_dir = '/content/drive/MyDrive/Jigsaw_Bias_Project/Models/bert_mitigated_final'

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Save the model weights and the tokenizer dictionary
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"SUCCESS: Model and Tokenizer permanently saved at: {output_dir}")

Saving the BERT Brain to Shared Drive...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SUCCESS: Model and Tokenizer permanently saved at: /content/drive/MyDrive/Jigsaw_Bias_Project/Models/bert_mitigated_final
